In [5]:
#Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
#Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
#Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content" : text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content" : text}
    messages.append(assistant_message)

def chat(messages, system = None, temperature = 1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    
    return message.content[0].text

In [ ]:
import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [7]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent = 2)

In [8]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [16]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [18]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }


In [ ]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""

    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])

    print(f"Average score:" {average_score})
    
    return results

In [19]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [20]:
print(json.dumps(results, indent=4))

[
    {
        "output": "# Solution: Extract AWS Account ID from ARN\n\nHere's a Python function that extracts the AWS account ID from an ARN string:\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS account ID from an ARN string.\n    \n    ARN format: arn:aws:service:region:account-id:resource\n    \n    Args:\n        arn (str): The ARN string\n        \n    Returns:\n        str: The AWS account ID (12-digit number)\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    parts = arn.split(':')\n    \n    if len(parts) < 5:\n        raise ValueError(f\"Invalid ARN format: {arn}\")\n    \n    account_id = parts[4]\n    \n    if not account_id:\n        raise ValueError(f\"Account ID is empty in ARN: {arn}\")\n    \n    return account_id\n```\n\n## Usage Examples\n\n```python\n# Test cases\ntest_arns = [\n    \"arn:aws:iam::123456789012:user/John\",\n    \"arn:aws:s3:::my-bucket\",\n    \"arn:aws:ec2:u